# Code-Aware Chunking

## Overview

This document explains how the debug assistant chunks Python source code for embedding — specifically, why code needs a different chunking strategy than prose documents, and how LangChain's `Language.PYTHON` splitter preserves logical boundaries like functions and classes.

## Architecture Context

The debug assistant indexes Python projects before debugging. The indexer (`indexer.py`) walks the project directory, chunks each `.py` file, embeds the chunks, and stores them in Qdrant. The agent then retrieves relevant chunks when answering questions like "why is `auth.py` failing?"

This is structurally identical to the ingestion service chunking PDFs — but code has structure that prose doesn't. A paragraph can be split almost anywhere without losing meaning. A function cannot: split a function in the middle and you lose the return statement, or worse, you embed a chunk whose meaning only makes sense with the surrounding context.

The ingestion service uses:
```
RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
```

The debug assistant uses:
```
RecursiveCharacterTextSplitter.from_language(Language.PYTHON, chunk_size=1500, chunk_overlap=200)
```

Same class. Different factory method. The `from_language()` call pre-configures the separator hierarchy to match Python's syntax instead of natural prose boundaries.

## Package Introductions

The document-qa notebooks already cover `langchain-text-splitters` and `RecursiveCharacterTextSplitter`. This section covers only what's new.

### `Language.PYTHON` and the `from_language()` factory

`RecursiveCharacterTextSplitter.from_language(Language.PYTHON)` is a factory method that returns a pre-configured splitter with separator patterns tuned for Python syntax.

Why does this exist? Prose text has natural split points: paragraphs (`\n\n`), then sentences (`.`, `?`, `!`), then words (` `). Code has a different hierarchy:

1. Class definitions (`\nclass ` — split between classes first)
2. Function definitions (`\ndef ` — then between functions)
3. Blank lines (`\n\n` — then between logical blocks)
4. Single newlines (`\n` — then between statements)
5. Individual characters (last resort)

This matters because splitting mid-function breaks semantic meaning worse than splitting mid-paragraph. If you embed a fragment that starts at `return` with no surrounding context, no retrieval query will find it usefully — the LLM needs to see the function signature to understand what's being returned.

The separators aren't AST-based — they're regex patterns that look for Python syntax. This means they're fast and dependency-free, but they can be fooled by unusual formatting. More on that in the Experiment section.

## Go/TS Comparison

| Concept | Go/TS | Python |
|---------|-------|--------|
| AST-based splitting | `go/ast`, `ts-morph` | `Language.PYTHON` splitter (regex-based, not true AST) |
| File walking | `filepath.WalkDir`, `fs.readdirSync` | `os.walk` with dir pruning |
| Path handling | `filepath.Rel` | `os.path.relpath` |

The biggest conceptual difference is that Python's `Language.PYTHON` splitter is **not** an AST parser. Go's `go/ast` and TypeScript's `ts-morph` parse source into a real syntax tree and can extract exact node boundaries. LangChain's language splitter uses regex patterns to approximate those boundaries — much simpler to use, but it can be wrong if your code formatting is unusual.

For an AST approach in Python you'd use the built-in `ast` module (`ast.parse()`, then walk the tree to find `FunctionDef` and `ClassDef` nodes and their `lineno`/`end_lineno` attributes). That would give exact boundaries, but it adds complexity. The regex approach is good enough for a debug assistant where chunk quality matters more than perfection.

## Build It

In [ ]:
# Prerequisites
# pip install langchain-text-splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter, Language
import tempfile, os, textwrap
print("Packages loaded!")

### Step 1: See the difference — generic vs language-aware splitting

The best way to understand why `Language.PYTHON` matters is to see both splitters on the same input. Let's create a Python file with a few small functions and chunk it both ways.

In [ ]:
SAMPLE_CODE = textwrap.dedent("""\
    import os
    from typing import Optional


    class AuthManager:
        """Handles user authentication."""

        def __init__(self, secret_key: str):
            self.secret_key = secret_key
            self._token_cache: dict[str, str] = {}

        def create_token(self, user_id: str) -> str:
            """Generate a new token for the given user."""
            import hashlib
            token = hashlib.sha256(f"{self.secret_key}:{user_id}".encode()).hexdigest()
            self._token_cache[user_id] = token
            return token

        def validate_token(self, user_id: str, token: str) -> bool:
            """Return True if token matches the cached token for user_id."""
            cached = self._token_cache.get(user_id)
            if cached is None:
                return False
            return cached == token


    def load_config(path: str) -> dict:
        """Load a JSON config file and return its contents as a dict."""
        import json
        if not os.path.exists(path):
            raise FileNotFoundError(f"Config not found: {path}")
        with open(path) as f:
            return json.load(f)


    def get_env(key: str, default: Optional[str] = None) -> Optional[str]:
        """Read an environment variable, returning default if not set."""
        return os.environ.get(key, default)
""")

print(f"Sample code: {len(SAMPLE_CODE)} characters, {SAMPLE_CODE.count(chr(10))} lines")

In [ ]:
# Generic splitter — the one the ingestion service uses for prose
generic_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
)

# Language-aware splitter — what the debug assistant uses
python_splitter = RecursiveCharacterTextSplitter.from_language(
    Language.PYTHON,
    chunk_size=300,
    chunk_overlap=50,
)

generic_chunks = generic_splitter.split_text(SAMPLE_CODE)
python_chunks = python_splitter.split_text(SAMPLE_CODE)

print(f"Generic splitter:  {len(generic_chunks)} chunks")
print(f"Python splitter:   {len(python_chunks)} chunks")
print()

print("=" * 60)
print("GENERIC CHUNKS")
print("=" * 60)
for i, chunk in enumerate(generic_chunks):
    first_line = chunk.strip().splitlines()[0] if chunk.strip() else "(empty)"
    print(f"[{i}] {len(chunk):>4}ch  starts: {first_line[:60]}")

print()
print("=" * 60)
print("PYTHON-AWARE CHUNKS")
print("=" * 60)
for i, chunk in enumerate(python_chunks):
    first_line = chunk.strip().splitlines()[0] if chunk.strip() else "(empty)"
    print(f"[{i}] {len(chunk):>4}ch  starts: {first_line[:60]}")

Look at what the Python-aware splitter does differently: it prefers to split **between** top-level definitions rather than mid-function. The generic splitter doesn't know that `def validate_token` belongs with `class AuthManager` — it just counts characters.

You can inspect the exact separators the Python splitter uses:

In [ ]:
# See the separator hierarchy Language.PYTHON uses
separators = RecursiveCharacterTextSplitter.get_separators_for_language(Language.PYTHON)
print("Python separator hierarchy (tried in order):")
for i, sep in enumerate(separators):
    display = repr(sep)
    print(f"  {i+1}. {display}")

### Step 2: Walk a directory collecting `.py` files

The indexer needs to walk an entire project, not just one file. Python's `os.walk` is the standard tool — it yields `(dirpath, dirnames, filenames)` tuples recursively.

The tricky part is **skipping directories we don't want** (`.git`, `__pycache__`, `venv`, etc.). `os.walk` lets you prune by modifying `dirs` **in-place** — this is a Python-specific pattern that doesn't have a direct Go/TS equivalent.

In [ ]:
SKIP_DIRS = {".git", "__pycache__", ".venv", "venv", "node_modules", ".mypy_cache", ".pytest_cache"}

def collect_python_files(root: str) -> list[str]:
    """Walk root and return relative paths of all .py files, skipping common non-source dirs."""
    found = []
    for dirpath, dirs, filenames in os.walk(root):
        # Prune dirs IN PLACE — this tells os.walk not to descend into them.
        # Assigning dirs[:] = [...] modifies the list object os.walk holds a reference to.
        # Assigning dirs = [...] would create a new list that os.walk never sees.
        dirs[:] = [d for d in dirs if d not in SKIP_DIRS]

        for filename in filenames:
            if filename.endswith(".py"):
                abs_path = os.path.join(dirpath, filename)
                rel_path = os.path.relpath(abs_path, root)
                found.append(rel_path)
    return sorted(found)


# Demo: create a temp project with .py files and some dirs to skip
with tempfile.TemporaryDirectory() as tmpdir:
    # Source files
    os.makedirs(os.path.join(tmpdir, "app"))
    os.makedirs(os.path.join(tmpdir, "app", "utils"))
    open(os.path.join(tmpdir, "app", "main.py"), "w").close()
    open(os.path.join(tmpdir, "app", "utils", "auth.py"), "w").close()
    open(os.path.join(tmpdir, "app", "utils", "config.py"), "w").close()

    # Dirs that should be skipped
    os.makedirs(os.path.join(tmpdir, "__pycache__"))
    open(os.path.join(tmpdir, "__pycache__", "main.cpython-311.pyc"), "w").close()
    os.makedirs(os.path.join(tmpdir, ".git"))
    open(os.path.join(tmpdir, ".git", "config"), "w").close()

    files = collect_python_files(tmpdir)
    print("Found .py files:")
    for f in files:
        print(f"  {f}")
    print(f"\n__pycache__ and .git were skipped — no .pyc files in results")

> **Pitfall:** `dirs[:] = [...]` and `dirs = [...]` look almost identical but do opposite things. The slice assignment `dirs[:] =` mutates the existing list in place — `os.walk` holds a reference to that same list object and reads it after your loop body runs to decide which subdirectories to recurse into. A plain `dirs =` creates a new local variable pointing to a new list; `os.walk` never sees it and happily descends into `__pycache__` anyway.

### Step 3: Track line numbers

The whole point of a debug assistant is to tell the user *where* a problem is. "The bug is in `auth.py`" is helpful. "The bug is in `auth.py` around line 42" is much better.

The `Language.PYTHON` splitter returns text chunks — it doesn't track which lines they came from. We need to recover that information ourselves.

The approach: after splitting, take the first line of each chunk and search for it in the original source lines. That gives us `start_line`. Count the chunk's own lines to get `end_line`.

In [ ]:
def _find_start_line(chunk: str, source_lines: list[str]) -> int:
    """Return the 1-indexed line number where chunk starts in source_lines.

    Matches on the first non-empty line of the chunk.
    Returns 1 if no match is found (safe fallback).
    """
    chunk_lines = chunk.splitlines()
    # Find first non-empty line to use as the search anchor
    anchor = next((ln.rstrip() for ln in chunk_lines if ln.strip()), None)
    if anchor is None:
        return 1

    for i, source_line in enumerate(source_lines):
        if source_line.rstrip() == anchor:
            return i + 1  # 1-indexed

    return 1  # fallback if not found


# Demo
source = SAMPLE_CODE
source_lines = source.splitlines()

python_splitter_demo = RecursiveCharacterTextSplitter.from_language(
    Language.PYTHON, chunk_size=400, chunk_overlap=50
)
chunks = python_splitter_demo.split_text(source)

print(f"{'Chunk':<6} {'start':>5} {'end':>5}  first line")
print("-" * 70)
for i, chunk in enumerate(chunks):
    start = _find_start_line(chunk, source_lines)
    end = start + chunk.count("\n")
    first_line = chunk.strip().splitlines()[0][:55] if chunk.strip() else "(empty)"
    print(f"{i:<6} {start:>5} {end:>5}  {first_line}")

> **Pitfall:** The line-matching approach can give wrong results if identical lines appear multiple times (e.g., multiple `return None` lines, multiple `pass` statements, or repeated docstring patterns). It always returns the *first* occurrence. For debugging purposes this is acceptable — line numbers are informational, not load-bearing. If the agent says "around line 42" and the actual occurrence is at line 87, the user can still search. A future improvement would be to use Python's `ast` module to get exact `FunctionDef.lineno` values, but that adds complexity for marginal gain.

### Step 4: Put it together — `chunk_code_files`

Now we combine file walking, language-aware splitting, and line tracking into the function the indexer actually calls.

In [ ]:
def chunk_code_files(
    root: str,
    chunk_size: int = 1500,
    chunk_overlap: int = 200,
) -> list[dict]:
    """Walk root, chunk every .py file with language-aware splitting.

    Returns a list of dicts, each with:
      text       - the chunk content
      file_path  - relative path from root
      start_line - 1-indexed line where chunk begins
      end_line   - 1-indexed line where chunk ends (approximate)
    """
    splitter = RecursiveCharacterTextSplitter.from_language(
        Language.PYTHON,
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )

    results = []
    py_files = collect_python_files(root)

    for rel_path in py_files:
        abs_path = os.path.join(root, rel_path)
        try:
            source = open(abs_path, encoding="utf-8").read()
        except (OSError, UnicodeDecodeError):
            # Skip files we can't read (binary files with .py extension, encoding issues)
            continue

        if not source.strip():
            continue  # Skip empty files

        source_lines = source.splitlines()
        chunks = splitter.split_text(source)

        for chunk in chunks:
            if not chunk.strip():
                continue
            start_line = _find_start_line(chunk, source_lines)
            end_line = start_line + chunk.count("\n")
            results.append({
                "text": chunk,
                "file_path": rel_path,
                "start_line": start_line,
                "end_line": end_line,
            })

    return results


# Demo: create a temp project and index it
with tempfile.TemporaryDirectory() as tmpdir:
    # Write our sample code to a file
    auth_path = os.path.join(tmpdir, "auth.py")
    with open(auth_path, "w") as f:
        f.write(SAMPLE_CODE)

    # Write a second file
    utils_path = os.path.join(tmpdir, "utils.py")
    with open(utils_path, "w") as f:
        f.write(textwrap.dedent("""\
            def add(a: int, b: int) -> int:
                return a + b


            def subtract(a: int, b: int) -> int:
                return a - b
        """))

    chunks = chunk_code_files(tmpdir, chunk_size=400, chunk_overlap=50)

    print(f"Total chunks: {len(chunks)}\n")
    for chunk in chunks:
        first_line = chunk["text"].strip().splitlines()[0][:55]
        print(
            f"{chunk['file_path']:<10}  "
            f"lines {chunk['start_line']:>3}–{chunk['end_line']:<3}  "
            f"{first_line}"
        )

## Experiment

### Experiment 1: How does `chunk_size` affect function boundaries?

The Python splitter *prefers* to split between functions, but if a function is larger than `chunk_size`, it has no choice but to split inside it. Let's see that threshold.

In [ ]:
# A file with many small functions — each fits easily in 300+ chars
many_small_functions = textwrap.dedent("""\
    def alpha() -> str:
        """Returns 'alpha'."""
        return "alpha"


    def beta() -> str:
        """Returns 'beta'."""
        return "beta"


    def gamma() -> str:
        """Returns 'gamma'."""
        return "gamma"


    def delta() -> str:
        """Returns 'delta'."""
        return "delta"


    def epsilon() -> str:
        """Returns 'epsilon'."""
        return "epsilon"
""")

print(f"Source: {len(many_small_functions)} chars, {many_small_functions.count(chr(10))} lines")
print()

for size in [500, 1500, 3000]:
    splitter = RecursiveCharacterTextSplitter.from_language(
        Language.PYTHON, chunk_size=size, chunk_overlap=0
    )
    chunks = splitter.split_text(many_small_functions)
    starts = [c.strip().splitlines()[0][:40] for c in chunks if c.strip()]
    print(f"chunk_size={size}: {len(chunks)} chunks")
    for s in starts:
        print(f"  starts: {s}")
    print()

### Experiment 2: What happens with a single large function?

If a function is longer than `chunk_size`, the splitter has no good split point — there's no `class ` or `def ` boundary inside it. It falls back to splitting on blank lines, then newlines. The overlap keeps some context across the boundary.

In [ ]:
# A single 200-line function — can't fit in a small chunk
lines = ["def big_processor(data: list[dict]) -> list[dict]:"]
lines.append('    """Process a large dataset with many transformations."""')
lines.append("    results = []")
for i in range(60):  # 60 iterations of similar statements
    lines.append(f"    # Step {i+1}: process item")
    lines.append(f"    value_{i} = data[{i}]['key'] if {i} < len(data) else None")
    lines.append(f"    if value_{i} is not None:")
    lines.append(f"        results.append({{'index': {i}, 'value': value_{i}}})")
    lines.append("")
lines.append("    return results")
big_function = "\n".join(lines)

print(f"Big function: {len(big_function)} chars, {big_function.count(chr(10))} lines")
print()

splitter = RecursiveCharacterTextSplitter.from_language(
    Language.PYTHON, chunk_size=500, chunk_overlap=100
)
chunks = splitter.split_text(big_function)

print(f"Chunks produced: {len(chunks)}")
print()

# Show where chunks start and end to see where splits happened
for i, chunk in enumerate(chunks):
    chunk_lines = chunk.splitlines()
    first = chunk_lines[0][:60] if chunk_lines else "(empty)"
    last = chunk_lines[-1][:60] if chunk_lines else "(empty)"
    print(f"Chunk {i}: {len(chunk)} chars")
    print(f"  first: {first}")
    print(f"  last:  {last}")
    print()

# Show the overlap in action between chunk 0 and chunk 1
if len(chunks) >= 2:
    end_of_0 = chunks[0].splitlines()[-3:]
    start_of_1 = chunks[1].splitlines()[:3]
    print("Overlap between chunk 0 and chunk 1:")
    print("  Last 3 lines of chunk 0:", end_of_0)
    print("  First 3 lines of chunk 1:", start_of_1)

Notice that when forced to split inside a function, the splitter falls back to blank-line boundaries inside the loop body. The `chunk_overlap=100` setting means the last 100 characters of chunk 0 also appear at the start of chunk 1 — so if the embedding for chunk 1 is retrieved, the LLM still sees some surrounding context from where the split happened.

## Check Your Understanding

1. **Why is 1500 characters used as the default `chunk_size` for code instead of the 1000 used for prose in the ingestion service?**

2. **The `Language.PYTHON` splitter uses regex patterns, not a true AST parser. What are the trade-offs of this approach?**

3. **Why do we store `start_line` and `end_line` as metadata on each chunk instead of just the text?**